# KhmerLive secure environment check\nVerify TLS Kafka, SILO object storage, PostgreSQL and MLflow from JupyterHub.

In [ ]:
import os, ssl\nimport urllib3\nfrom minio import Minio\nfrom confluent_kafka import Consumer\nfrom sqlalchemy import create_engine, text\nprint('Kafka:', os.environ['KAFKA_BOOTSTRAP_SERVERS'])\nprint('SILO:', os.environ['SILO_ENDPOINT'])\nprint('MLflow:', os.environ['MLFLOW_TRACKING_URI'])\n

In [ ]:
engine=create_engine(os.environ['DATABASE_URL'])\nwith engine.connect() as conn:\n    print(conn.execute(text('select version()')).scalar())\n

In [ ]:
pool=urllib3.PoolManager(cert_reqs=ssl.CERT_REQUIRED, ca_certs=os.environ['SILO_CA_FILE'])\nsilo=Minio(os.environ['SILO_ENDPOINT'], access_key=os.environ['SILO_ACCESS_KEY'], secret_key=os.environ['SILO_SECRET_KEY'], secure=True, http_client=pool)\nprint('SILO buckets:', [b.name for b in silo.list_buckets()])\n

In [ ]:
cfg={'bootstrap.servers':os.environ['KAFKA_BOOTSTRAP_SERVERS'],'group.id':'jupyter-env-check','auto.offset.reset':'latest','security.protocol':os.environ.get('KAFKA_SECURITY_PROTOCOL','SSL'),'ssl.ca.location':os.environ['KAFKA_SSL_CA_LOCATION']}\nconsumer=Consumer(cfg)\nprint('Kafka topics:', sorted(consumer.list_topics(timeout=5).topics))\nconsumer.close()\n